# Phase 1: Basic Loading & Trimming
In this phase we load the data and remove garbage. This is perfectly safe to do before Train/Test Split because it does not involve calculating any statistics that could cause data leakage.

In [1]:
import pandas as pd 
import numpy as np 
df = pd.read_csv("/Users/nishchaljain/Downloads/dataset.csv")

# 1. Print Shape and check head
print("Original Shape:", df.shape)
print("Data loaded successfully!")

# 2. Fix typos in categorical values
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
for col in categorical_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False)) 
    print("-" * 30 + "\n")

df['MSZoning'] = df['MSZoning'].replace('C (all)', 'C')
df['Exterior2nd'] = df['Exterior2nd'].replace({'CmentBd': 'CemntBd', 'Brk Cmn': 'BrkComm', 'Wd Shng': 'WdShing'})

# 3. Drop normal duplicates
df = df.drop_duplicates()

# 4. Drop near duplicates - What details never change that we can use to perfectly identify this exact thing
subset_cols = ['LotArea', 'YearBuilt', 'Neighborhood', 'BldgType', 'GrLivArea', 'GarageArea']
df = df.drop_duplicates(subset=subset_cols, keep='first')

# 5. Drop columns with 50%+ missing values
missing_percentages = df.isnull().sum() / len(df)
cols_to_drop = missing_percentages[missing_percentages > 0.5].index   
df = df.drop(columns=cols_to_drop)

# 6. Drop columns with 99%+ identical values
useless_cols = [c for c in df.columns if df[c].value_counts(normalize=True).iloc[0] > 0.99]
df = df.drop(columns=useless_cols)

# 7. Prevent Target Leakage by At the exact moment I need this model to make a real-world prediction, will this piece of data actually exist yet
leakage_cols = ['MoSold','YrSold','SaleType','SaleCondition']
df = df.drop(columns=[c for c in leakage_cols if c in df.columns])
print("Shape after trimming:", df.shape)


Original Shape: (1460, 81)
Data loaded successfully!
--- MSZoning ---
MSZoning
RL         1151
RM          218
FV           65
RH           16
C (all)      10
Name: count, dtype: int64
------------------------------

--- Street ---
Street
Pave    1454
Grvl       6
Name: count, dtype: int64
------------------------------

--- Alley ---
Alley
NaN     1369
Grvl      50
Pave      41
Name: count, dtype: int64
------------------------------

--- LotShape ---
LotShape
Reg    925
IR1    484
IR2     41
IR3     10
Name: count, dtype: int64
------------------------------

--- LandContour ---
LandContour
Lvl    1311
Bnk      63
HLS      50
Low      36
Name: count, dtype: int64
------------------------------

--- Utilities ---
Utilities
AllPub    1459
NoSeWa       1
Name: count, dtype: int64
------------------------------

--- LotConfig ---
LotConfig
Inside     1052
Corner      263
CulDSac      94
FR2          47
FR3           4
Name: count, dtype: int64
------------------------------

--- LandSlop

# Phase 2: The Vault (Train/Test Split)
**CRITICAL INDUSTRY STANDARD**: We must split our data into Training and Testing sets **BEFORE** we perform Imputation, ANOVA, Scaling, or Outlier Capping. If we calculate the 'mean' on the entire dataset, the test set's mean leaks into our training! That's Data Leakage. From here on out, we put the test set in a vault and only calculate statistics on the `X_train`!

In [2]:
from sklearn.model_selection import train_test_split

target_col = 'SalePrice'
X = df.drop(columns=[target_col])
y = df[target_col]

# Split 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")


Training Data Shape: (1156, 68)
Testing Data Shape: (290, 68)


# Phase 3: Imputation (Fixing Missing Values)
Notice how we calculate the median/mean using ONLY `X_train`, but we `.fillna()` on both `X_train` and `X_test`!

In [3]:
# Categorical: if missing value means not exisiting then just fill it with none orelse fill it with mode
for col in X_train.select_dtypes('object'):
    X_train[col] = X_train[col].fillna('None')
    X_test[col] = X_test[col].fillna('None')

# Numeric: Fill with Median or Mean calculated strictly from X_train
for col in X_train.select_dtypes(['int64', 'float64']):
    if abs(X_train[col].skew()) > 1:
        impute_val = X_train[col].median()
    else:
        impute_val = X_train[col].mean()
        
    X_train[col] = X_train[col].fillna(impute_val)
    X_test[col] = X_test[col].fillna(impute_val)

print("Missing values fixed without Data Leakage!")


Missing values fixed without Data Leakage!


# Phase 4: Feature Selection (Math Tests)
We run correlation and ANOVA tests against `y_train` using ONLY `X_train`.

In [4]:
import scipy.stats as stats

# for numeric columns we make decisions based on corelation 
numeric_X = X_train.select_dtypes(include=[np.number])

# A. Feature-to-Feature Correlation
corr_matrix = numeric_X.corr() 
high_corr_pairs = []
threshold = 0.80

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        corr_value = abs(corr_matrix.iloc[i, j])
        if corr_value > threshold:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j] , corr_value))

print("--- Highly Correlated Pairs ---")
for pair in high_corr_pairs:
    print(f"{pair[0]} and {pair[1]} : {pair[2]:.3f}")
print("\n")

print("--- Correlation with Target (SalePrice) ---")
target_corr = numeric_X.corrwith(y_train).abs().sort_values(ascending=False)
print(target_corr.head(10)) # Just printing the top 10 so it doesn't flood your console
print("\n")

# C. Drop the weaker twins 
cols_to_drop_multi = ['GarageArea', 'TotRmsAbvGrd','TotRmsAbvGrd'] 
X_train = X_train.drop(columns=cols_to_drop_multi, errors='ignore')
X_test = X_test.drop(columns=cols_to_drop_multi, errors='ignore')


# ==========================================
# 2. TEXT FEATURES: ANOVA Test 
# ==========================================
# We temporarily merge HERE because grouping text categories (like Neighborhoods) 
# against the target prices is much easier when they are in the same dataframe.
train_temp = pd.concat([X_train, y_train], axis=1)
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

anova_scores = []
dead_weight_text = []

for col in cat_cols:
    # Group the target (SalePrice) by the unique categories in the current text column
    groups = [train_temp[train_temp[col] == cat]['SalePrice'].dropna() for cat in train_temp[col].unique()]
    
    # We need at least 2 groups to run an ANOVA test
    if len(groups) > 1:
        f_stat, p_val = stats.f_oneway(*groups)
        anova_scores.append((col, p_val))

# Filter for the useless columns
for col, p in anova_scores:
    if p > 0.05: # A p-value > 0.05 means this column doesn't significantly impact the price
        dead_weight_text.append(col)

print(f"Dropping statistically weak text columns: {dead_weight_text}")
X_train = X_train.drop(columns=dead_weight_text, errors='ignore')
X_test = X_test.drop(columns=dead_weight_text, errors='ignore')

"""
# ==========================================
# 2. TEXT FEATURES (CLASSIFICATION): Chi-Square Test (between text to text)(if our target is a text)
# ==========================================
# Temporarily merge X_train and y_train just like before
train_temp = pd.concat([X_train, y_train], axis=1)

# Get the name of your target column directly from y_train
target_col = y_train.name 

cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

chi2_scores = []
dead_weight_text = []

for col in cat_cols:
    # 1. Create a contingency table (counts how many times each feature category 
    #    overlaps with each target category)
    contingency_table = pd.crosstab(train_temp[col], train_temp[target_col])
    
    # 2. Run the Chi-Square test
    # chi2_contingency returns 4 things, but we only care about the p-value (index 1)
    chi2_stat, p_val, dof, expected = stats.chi2_contingency(contingency_table)
    
    chi2_scores.append((col, p_val))

# 3. Filter for the useless columns
for col, p in chi2_scores:
    # A p-value > 0.05 means the feature and target are independent (the feature 
    # doesn't give us any clues about the target class).
    if p > 0.05: 
        dead_weight_text.append(col)

print(f"Dropping statistically weak text columns: {dead_weight_text}")
X_train = X_train.drop(columns=dead_weight_text, errors='ignore')
X_test = X_test.drop(columns=dead_weight_text, errors='ignore')
"""


--- Highly Correlated Pairs ---
TotalBsmtSF and 1stFlrSF : 0.809
GrLivArea and TotRmsAbvGrd : 0.827
GarageCars and GarageArea : 0.882


--- Correlation with Target (SalePrice) ---
OverallQual     0.791631
GrLivArea       0.726901
GarageCars      0.639641
GarageArea      0.621825
TotalBsmtSF     0.613187
1stFlrSF        0.606667
FullBath        0.568927
TotRmsAbvGrd    0.541488
YearBuilt       0.521669
YearRemodAdd    0.493198
dtype: float64


Dropping statistically weak text columns: ['LandSlope', 'Condition2']


'\n# ==========================================\n# 2. TEXT FEATURES (CLASSIFICATION): Chi-Square Test (between text to text)(if our target is a text)\n# ==========================================\n# Temporarily merge X_train and y_train just like before\ntrain_temp = pd.concat([X_train, y_train], axis=1)\n\n# Get the name of your target column directly from y_train\ntarget_col = y_train.name \n\ncat_cols = X_train.select_dtypes(include=[\'object\', \'category\']).columns\n\nchi2_scores = []\ndead_weight_text = []\n\nfor col in cat_cols:\n    # 1. Create a contingency table (counts how many times each feature category \n    #    overlaps with each target category)\n    contingency_table = pd.crosstab(train_temp[col], train_temp[target_col])\n    \n    # 2. Run the Chi-Square test\n    # chi2_contingency returns 4 things, but we only care about the p-value (index 1)\n    chi2_stat, p_val, dof, expected = stats.chi2_contingency(contingency_table)\n    \n    chi2_scores.append((col, p_val)

# Phase 5: Outliers & Skewness
**CRITICAL RULE**: We NEVER cap our target variable `y_train`! We only cap `X_train` and `X_test` using `X_train`'s boundaries. Then we apply `np.log1p()` to skewed features, including `y_train` and `y_test`!

In [5]:
# 1. Hybrid Capping (Winsorization) ONLY on Features!
vip_features = ['GrLivArea', 'LotArea', 'TotalBsmtSF', '1stFlrSF']
numeric_cols = [c for c in X_train.select_dtypes(["int64", "float64"]).columns]

for col in numeric_cols:
    # Calculate boundaries using ONLY the training data
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    if col in vip_features:
        X_train[f'Is_Massive_{col}'] = (X_train[col] > upper_bound).astype(int)
        X_test[f'Is_Massive_{col}'] = (X_test[col] > upper_bound).astype(int)
        
    X_train[col] = X_train[col].clip(lower=lower_bound, upper=upper_bound)
    X_test[col] = X_test[col].clip(lower=lower_bound, upper=upper_bound)

# 2. Categorical Normalization (Club rare values to 'Other' based on Train distribution)
for col in X_train.select_dtypes('object'):
    freq = X_train[col].value_counts(normalize=True)
    rare_categories = freq[freq < 0.01].index
    X_train[col] = X_train[col].replace(rare_categories, 'Other')
    X_test[col] = X_test[col].replace(rare_categories, 'Other')

# 3. Log Transform Skewed Features
numeric_cols = [c for c in X_train.select_dtypes(["int64", "float64"]).columns if not c.startswith('Is_Massive_')]
for col in numeric_cols:
    if abs(X_train[col].skew()) > 1:
        X_train[col] = np.log1p(X_train[col])
        X_test[col] = np.log1p(X_test[col])

# 4. Log Transform the Target Variables (Because housing prices are skewed)
y_train = np.log1p(y_train)
y_test = np.log1p(y_test)

print("Features securely capped and transformed!")

Features securely capped and transformed!


# Phase 6: Encoding & Super Features

In [6]:
from sklearn.preprocessing import OneHotEncoder, TargetEncoder

print("🚨 EXECUTING INDUSTRY-STANDARD HYBRID ENCODING (PURE SKLEARN) 🚨")

cat_cols = X_train.select_dtypes(['object', 'category']).columns

# Target Encoding
high_card_cols = [c for c in cat_cols if X_train[c].nunique() > 10]
te = TargetEncoder(target_type="continuous") 
te.fit(X_train[high_card_cols], y_train)
train_te_df = pd.DataFrame(te.transform(X_train[high_card_cols]), columns=high_card_cols, index=X_train.index)
test_te_df = pd.DataFrame(te.transform(X_test[high_card_cols]), columns=high_card_cols, index=X_test.index)

# One-Hot Encoding
low_card_cols = [c for c in cat_cols if X_train[c].nunique() <= 10]
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') 
ohe.fit(X_train[low_card_cols])
train_ohe_df = pd.DataFrame(ohe.transform(X_train[low_card_cols]), columns=ohe.get_feature_names_out(low_card_cols), index=X_train.index)
test_ohe_df = pd.DataFrame(ohe.transform(X_test[low_card_cols]), columns=ohe.get_feature_names_out(low_card_cols), index=X_test.index)

# Extract final numerics
numeric_cols = X_train.select_dtypes(['int64', 'float64']).columns
X_train_final = pd.concat([X_train[numeric_cols], train_te_df, train_ohe_df], axis=1)
X_test_final = pd.concat([X_test[numeric_cols], test_te_df, test_ohe_df], axis=1)

# Check 24: Engineering Super Features
for data in [X_train_final, X_test_final]:
    data['Total_SF'] = data.get('TotalBsmtSF', 0) + data.get('1stFlrSF', 0) + data.get('2ndFlrSF', 0)
    data['Total_Baths'] = data.get('FullBath', 0) + (0.5 * data.get('HalfBath', 0)) + data.get('BsmtFullBath', 0)
    data['House_Age'] = 2010 - data['YearBuilt']

# 2. Drop the original constituent features to prevent multicollinearity
cols_to_drop_super = [
    'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 
    'FullBath', 'HalfBath', 'BsmtFullBath', 
    'YearBuilt'
]

# Using errors='ignore' just in case a column was already dropped earlier in the pipeline
X_train_final = X_train_final.drop(columns=cols_to_drop_super, errors='ignore')
X_test_final = X_test_final.drop(columns=cols_to_drop_super, errors='ignore')
print(f"✅ Success! X_train final shape after dropping originals: {X_train_final.shape}")

🚨 EXECUTING INDUSTRY-STANDARD HYBRID ENCODING (PURE SKLEARN) 🚨
✅ Success! X_train final shape after dropping originals: (1156, 165)


# Phase 7: Scaling
Scale our numeric columns safely (excluding flags) using standard scaler.

In [7]:
# Standard Scaling (Z-Score)
cols_to_scale = [c for c in X_train_final.columns if not c.startswith('Is_Massive_')]

# Drop zero variance columns to get rid of divide by 0 problem
zero_variance_cols = [col for col in cols_to_scale if X_train_final[col].std() == 0]
X_train_final = X_train_final.drop(columns=zero_variance_cols)
X_test_final = X_test_final.drop(columns=zero_variance_cols)
cols_to_scale = [c for c in cols_to_scale if c not in zero_variance_cols]

# We fit our scaler manually using X_train's mean and std
for col in cols_to_scale:
    train_mean = X_train_final[col].mean()
    train_std = X_train_final[col].std()
    
    X_train_final[col] = (X_train_final[col] - train_mean) / train_std
    X_test_final[col] = (X_test_final[col] - train_mean) / train_std

print("Scaling complete. Data is mathematically ready!")


Scaling complete. Data is mathematically ready!


# Phase 8: finding best feature
Train a Random Forest, scan the top 10 features, and test its accuracy!

In [8]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

print("🌲 RANDOM FOREST IMPORTANCE SCAN 🌲")

# 1. Initialize and train a basic Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_final, y_train)

# 2. Extract the importance scores
importance_df = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n🏆 THE TOP 10 MOST POWERFUL FEATURES 🏆")
print("-" * 40)
print(importance_df.head(10).to_string(index=False))

# 3. Extract the names of the top 10 features into a list
top_10_features = importance_df['Feature'].head(10).tolist()

# 4. Overwrite standard X_train and X_test for the downstream pipeline
# (Passing a list of columns to a DataFrame automatically returns a new DataFrame)
X_train = X_train_final[top_10_features]
X_test = X_test_final[top_10_features]

print(f"\n✅ Pipeline Ready! X_train is a {type(X_train).__name__} with shape: {X_train.shape}")

🌲 RANDOM FOREST IMPORTANCE SCAN 🌲

🏆 THE TOP 10 MOST POWERFUL FEATURES 🏆
----------------------------------------
     Feature  Importance
 OverallQual    0.415188
    Total_SF    0.346027
Neighborhood    0.055312
   GrLivArea    0.014139
     LotArea    0.011884
 OverallCond    0.010579
   House_Age    0.010165
  BsmtFinSF1    0.009745
YearRemodAdd    0.008869
   BsmtUnfSF    0.008285

✅ Pipeline Ready! X_train is a DataFrame with shape: (1156, 10)


# figuring out which metrics to use for evaluation 

In [9]:
'''
if our target variable was a text (classification) then we would have to look at the skewness of the target variable 
and then according to that figure out which metrics will be good for out ml model

# (Assuming your target column is called 'target_col')
target_col = 'Is_Fraud' 

# 1. Count the categories and get the percentages
vc = df[target_col].value_counts()
imbalance_ratio = vc.max() / vc.min()
print(f"\nImbalance Ratio: {imbalance_ratio:.1f} to 1")

# 3. The Metric Gates (Strict rules on how to grade the model)
if imbalance_ratio < 3:
    print("✅ Status: Balanced.")
    print("➡️ Metric to use: Accuracy is acceptable, but check F1-Score too.")

elif 3 <= imbalance_ratio <= 10:
    print("⚠️ Status: Moderate Imbalance.")
    print("➡️ Metric to use: DO NOT USE ACCURACY. Use F1-Score, Precision, and Recall.")
    print("➡️ Fix: Use `class_weight='balanced'` inside your model.") for example model = RandomForestClassifier(class_weight='balanced')

else:
    print("🚨 Status: Severe Imbalance (>10:1).")
    print("➡️ Metric to use: PR-AUC (Precision-Recall Area Under Curve).")
    print("➡️ Fix: You must use advanced fixes like SMOTE (creating fake minority data).")
'''



'''
because right now our target variable is number (regression) 
we use all the three 

Condition A: Normal Business Use

Focus on: MAE. It is the only metric a normal business stakeholder understands ("Our model is off by $15,000 on average").

Condition B: High-Risk Situations (e.g., Medical dosing, Trading algorithms)

Focus on: RMSE. You want the math to panic if the model is ever wildly wrong, because being wildly wrong is dangerous.

Condition C: The Baseline Context

Focus on: R². Use this to prove the model is actually learning and not just guessing the average every time.

We are going to make our models spit out all three. We will use R² to see who is the smartest overall, MAE to see the dollar-amount error, and RMSE to see who is failing the hardest on the expensive houses.
'''

'\nbecause right now our target variable is number (regression) \nwe use all the three \n\nCondition A: Normal Business Use\n\nFocus on: MAE. It is the only metric a normal business stakeholder understands ("Our model is off by $15,000 on average").\n\nCondition B: High-Risk Situations (e.g., Medical dosing, Trading algorithms)\n\nFocus on: RMSE. You want the math to panic if the model is ever wildly wrong, because being wildly wrong is dangerous.\n\nCondition C: The Baseline Context\n\nFocus on: R². Use this to prove the model is actually learning and not just guessing the average every time.\n\nWe are going to make our models spit out all three. We will use R² to see who is the smartest overall, MAE to see the dollar-amount error, and RMSE to see who is failing the hardest on the expensive houses.\n'